In [53]:
import pandas as pd
import numpy as np
jobs=pd.read_csv('data/fake_job_postings.csv')

In [54]:
jobs['fraudulent'].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [55]:
indices_to_drop = jobs[jobs['fraudulent'] == 0].sample(n=16000).index #fixing class imbalance
jobs = jobs.drop(indices_to_drop)

In [56]:
jobs['fraudulent'].value_counts()

fraudulent
0    1014
1     866
Name: count, dtype: int64

**Commitments made in the project plan**

1. Methods: Logistic Regression, SVM, RNN, Random Forest, BERT Transformer
2. Utilize the same or similar pre-processing technique
3. I will then train and test each model, evaluating
accuracy, precision, recall, and the macro-averaged F1 score and utilizing CodeCarbon’s ability to
estimate CO2 Emissions to log and obtain the emissions from training the data and report the calculated
CE_Rel and delta CE_rel metrics.

In [57]:
jobs.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
12,13,"Applications Developer, Digital","US, CT, Stamford",NaN,NaN,"Novitex Enterprise Solutions, formerly Pitney ...","The Applications Developer, Digital will devel...",Requirements:4 – 5 years’ experience in develo...,NaN,0,1,0,Full-time,Associate,Bachelor's Degree,Management Consulting,Information Technology,0
23,24,"Vice President, Sales and Sponsorship (Busines...","US, CA, Carlsbad",Businessfriend.com,100000-120000,"WDM Group is an innovative, forward thinking d...",#URL_eda2500ddcedb60957fcd7f5b164e092966f8c4e8...,"Job Requirements:A reputation as a ""go-getter""...",Businessfriend will offer a competitive six fi...,0,1,0,Full-time,Executive,Unspecified,Internet,Sales,0
46,47,Entry Level,"EG, C,",NaN,NaN,History &amp; Background The Bank started its ...,We offer diversified opportunities in various ...,-0-1 years experience English Bachelor in comm...,NaN,0,0,0,Full-time,Entry level,Bachelor's Degree,Banking,NaN,0
56,57,Outside Sales Professional-Oronoco,"US, MN, Oronoco",NaN,NaN,"ABC Supply Co., Inc. is the nation’s largest w...","As an Outside Sales Representative, you must h...",Track Record of Sales Success – B2B or B2CNo m...,"As an Outside Sales Representative, you will r...",0,1,0,NaN,NaN,NaN,NaN,NaN,0
61,62,Bulk Ingredient Unloader,"US, IA, Cedar Rapids",General Services,NaN,"Red Star Yeast Company LLC (RSYC), a leader in...",The primary function of this position is to pe...,Must be able to understand and follow the flow...,"Benefit, Compensation and Shift Schedule Detai...",0,1,1,Full-time,Entry level,High School or equivalent,Food Production,Supply Chain,0


In [58]:
model_df = jobs[['description', 'fraudulent']]

In [59]:
model_df.head()


,description,fraudulent
12,"The Applications Developer, Digital will devel...",0
23,#URL_eda2500ddcedb60957fcd7f5b164e092966f8c4e8...,0
46,We offer diversified opportunities in various ...,0
56,"As an Outside Sales Representative, you must h...",0
61,The primary function of this position is to pe...,0


In [60]:
import nltk
import re
import html
from nltk.corpus import stopwords
nltk.download('stopwords',quiet=True)
stop_words=set(stopwords.words('english'))


def preprocess_text(text):
  text=re.sub(r"(?:http\S+|@)","",text) #arguments are pattern,replace,string.
  text=html.unescape(text) #convert XML to string. This function can handle XML entities like &amp
  tokens=text.split() #just using split()
  tokens=[token for token in tokens if token not in stop_words]
  return ' '.join(tokens)

there's a class imbalance. much more non-fraudulent ones than fraudulent. There are ~17800 observations and maybe 800 fraudulent ones. Maybe consider fixing this (but don't think it has a direct effect on workload, but maybe want things to be realistic).

**LOGISTIC REGRESSION**

In [ ]:
##CHECK LAB 6 FOR K FOLD cross validation applied to logistic regression

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

In [62]:
model_df['description']=model_df['description'].astype(str)

/var/folders/tl/0pf8lcrd691gwn8392x3hs6r0000gn/T/ipykernel_969/3832600678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df['description']=model_df['description'].astype(str)


In [63]:
processed_descriptions=[preprocess_text(doc) for doc in model_df['description']]

In [64]:
labels=model_df['fraudulent']

In [18]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(processed_descriptions,labels,test_size=0.3,random_state=42)


In [19]:
tfidf=TfidfVectorizer()
X_tfidf = tfidf.fit_transform(x_train)
X_test_tfidf = tfidf.transform(x_test)

In [20]:
reg_classifier=LogisticRegression()
reg_classifier.fit(X_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [21]:
y_pred = reg_classifier.predict(X_test_tfidf) #should break to test train

In [22]:
print(classification_report(y_test, y_pred)) #ofc it did perfectly bc i didnt split it into test.train.

              precision    recall  f1-score   support

           0       0.84      0.93      0.88       307
           1       0.91      0.79      0.84       257

    accuracy                           0.87       564
   macro avg       0.87      0.86      0.86       564
weighted avg       0.87      0.87      0.87       564



**SUPPORT VECTOR MACHINE**

In [23]:
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.base import TransformerMixin
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [24]:
class SparsetoDense(TransformerMixin):
  def fit(self, x, y = None, **fit_params):
    return self
  def transform(self, x, y=None, **fit_params):
    return x.toarray()

In [25]:
svm_pipe=Pipeline([
    ('densify',SparsetoDense()),
    ('scale', StandardScaler()),
    ('classify',SVC())
])

kernel= ['rbf', 'linear']
C = [0.001, 0.01, 1, 10] #if its running too long will cut down on some of these combos
svm_params = {
    'classify__kernel': kernel,
    'classify__C': C
}

In [26]:
inner_cv=KFold(n_splits=3, shuffle=True, random_state=1)
outer_cv=KFold(n_splits=5, shuffle=True, random_state=1)

grid_SVC=GridSearchCV(svm_pipe, svm_params, cv=inner_cv)

In [27]:
scores=cross_validate(grid_SVC,
                     X=X_tfidf,
                     y=y_train,
                     cv=outer_cv,
                     scoring=['accuracy','f1','precision','recall'],
                     return_estimator=True)

In [28]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.86363636 0.86311787 0.85171103 0.8365019  0.83269962]
[0.85981308 0.87692308 0.83333333 0.86725664 0.79844961]
[0.81415929 0.85074627 0.82608696 0.77777778 0.85123967]
[0.83636364 0.86363636 0.82969432 0.82008368 0.824     ]


In [ ]:
#SVM Cross validation ran for 24 mins (wait it was faster locally on m4)

In [29]:
grid_SVC.fit(X_tfidf,y_train)
grid_SVC.best_params_

{'classify__C': 0.01, 'classify__kernel': 'linear'}

In [30]:
#bestmodel as SVC object
#then do bestmodel.fit
#then get y_pred with bestmodel
SVC_model = SVC(kernel='linear', C=0.001)
SVC_model.fit(X_tfidf, y_train)
y_pred = SVC_model.predict(X_test_tfidf)

In [31]:
#get classifciation report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.54      1.00      0.70       307
           1       0.00      0.00      0.00       257

    accuracy                           0.54       564
   macro avg       0.27      0.50      0.35       564
weighted avg       0.30      0.54      0.38       564



/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [32]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[307   0]
 [257   0]]


**RECURRENT NEURAL NET (RNN)**

In [33]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [40]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import metrics
from keras.layers import Dense,Input, GlobalMaxPooling1D, Dropout
from keras.layers import Conv1D, MaxPooling1D, Embedding, LSTM, SimpleRNN
from keras.models import Model, Sequential
from keras.initializers import Constant

In [41]:
MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 300
VALIDATION_SPLIT = 0.2
EMBEDDING_DIM = 100 #going to use GLOVE if we end up doing embeddings

In [42]:
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS)
tokenizer.fit_on_texts(x_train)
train_sequences = tokenizer.texts_to_sequences(x_train)
test_sequences = tokenizer.texts_to_sequences(x_test)
word_index = tokenizer.word_index

In [43]:
trainvalid_data = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH)
trainvalid_labels = to_categorical(y_train, num_classes = 2)
test_labels = to_categorical(y_test, num_classes = 2) 

In [44]:
#think the below section should be moved to the top as we prob want to use GLOVE embeddings for everything

In [45]:
#time to split into train and validation
indices = np.arange(trainvalid_data.shape[0])
np.random.shuffle(indices)
trainvalid_data = trainvalid_data[indices]
trainvalid_labels = trainvalid_labels[indices]
num_validation_samples = int(0.2 * trainvalid_data.shape[0])
x_train = trainvalid_data[:-num_validation_samples]
y_train = trainvalid_labels[:-num_validation_samples]
x_val = trainvalid_data[-num_validation_samples:]
y_val = trainvalid_labels[-num_validation_samples:]

In [46]:
glove_dir = '../glove.6B.100d.txt'
glove_index = {}
with open(glove_dir, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        embeddings = np.asarray(values[1:], dtype='float32')
        glove_index[word]=embeddings

In [47]:
#making the embedding matrix
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
for word, i in word_index.items():
    if i > MAX_NUM_WORDS:
        continue
    embedding_vector = glove_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    

In [52]:
embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer = Constant(embedding_matrix),
                            input_length = MAX_SEQUENCE_LENGTH,
                            trainable=False)


/Users/ianch/miniconda3/envs/tf_m4/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [63]:
rnnmodel = Sequential()
rnnmodel.add(embedding_layer)
rnnmodel.add(LSTM(128, dropout = 0.25))
rnnmodel.add(Dense(2, activation = 'softmax')) #len(labels_index)

rnnmodel.compile(loss='categorical_crossentropy',
                 optimizer = 'Adam',
                 metrics = ['acc', metrics.Precision(), metrics.Recall(), metrics.F1Score(average='macro')])

In [64]:
tf.debugging.set_log_device_placement(True)
rnn_train = rnnmodel.fit(x_train, y_train,
                         batch_size = 16,
                         epochs = 3,
                         validation_data = (x_val, y_val))

Epoch 1/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - acc: 0.5973 - f1_score: 0.5904 - loss: 0.6626 - precision_2: 0.5973 - recall_2: 0.5973 - val_acc: 0.7300 - val_f1_score: 0.7290 - val_loss: 0.5781 - val_precision_2: 0.7300 - val_recall_2: 0.7300
Epoch 2/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - acc: 0.6952 - f1_score: 0.6901 - loss: 0.5848 - precision_2: 0.6952 - recall_2: 0.6952 - val_acc: 0.7110 - val_f1_score: 0.7095 - val_loss: 0.5619 - val_precision_2: 0.7110 - val_recall_2: 0.7110
Epoch 3/3
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - acc: 0.7569 - f1_score: 0.7546 - loss: 0.5148 - precision_2: 0.7569 - recall_2: 0.7569 - val_acc: 0.7110 - val_f1_score: 0.6978 - val_loss: 0.5516 - val_precision_2: 0.7110 - val_recall_2: 0.7110


In [ ]:
#recurrent_dropout > 0 majorly messed up performance and caused >1 min/step. 
#claude stated this parameter would cause it to fall back to CPU (and even though 
#i didnt see it in the messages, the insane increase in processing time when i removed
#this setting made me suspicious that it was silently falling back to cpu. 

In [65]:
loss, test_acc, test_precision, test_recall, test_f1 = rnnmodel.evaluate(test_data, test_labels)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - acc: 0.7358 - f1_score: 0.7226 - loss: 0.5590 - precision_2: 0.7358 - recall_2: 0.7358


In [66]:
print(f'Test data accuracy {test_acc} \n',
      f'Test data precision {test_precision} \n',
      f'Test data Recall {test_recall} \n',
      f'Test data F1 Score {test_f1}')

Test data accuracy 0.7358155846595764 
 Test data precision 0.7358155846595764 
 Test data Recall 0.7358155846595764 
 Test data F1 Score 0.7226232290267944


**RANDOM FOREST**

In [97]:
from sklearn.ensemble import RandomForestClassifier

In [114]:
desclist = list(processed_descriptions)
tokenized = []
for doc in desclist:
    sent = doc.lower()
    sent = doc.split()
    tokenized.append(sent)


In [122]:
sentence_embedding_list = []
for sentence in tokenized:
    word_embeddings = [glove_index.get(word) if word in glove_index.keys() else np.zeros(100) for word in sentence]
    if np.sum(word_embeddings)==0:
        sentence_embeddings = np.zeros(100)
    else:
        sentence_embeddings = np.mean(word_embeddings, axis = 0)
    sentence_embedding_list.append(sentence_embeddings)

In [126]:
np.array(sentence_embedding_list) #this should work if the loop ran correctly

array([[-0.12597807,  0.13993321,  0.0185024 , ..., -0.14703711,
         0.44409125,  0.16294534],
       [-0.01549626,  0.08979446,  0.01896285, ..., -0.19430406,
         0.5236805 ,  0.216092  ],
       [-0.044257  ,  0.20607056, -0.08313811, ..., -0.11686583,
         0.40525723,  0.17421933],
       ...,
       [-0.0580673 ,  0.08066092,  0.12535938, ...,  0.1104    ,
         0.21679308,  0.09770462],
       [-0.12449407,  0.11362701, -0.08407349, ..., -0.2438132 ,
         0.54900138,  0.27324951],
       [-0.14515872,  0.09980639, -0.02444845, ..., -0.18500703,
         0.44085038,  0.09799469]])

In [127]:
X_train, X_test, y_train, y_test = train_test_split(sentence_embedding_list, labels, 
                                                    test_size = 0.2,
                                                    random_state=1)

In [128]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [130]:
y_pred = rf.predict(X_test)

In [132]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87       217
           1       0.83      0.79      0.81       159

    accuracy                           0.84       376
   macro avg       0.84      0.84      0.84       376
weighted avg       0.84      0.84      0.84       376



**BERT Transformer**